# Imports

In [1]:

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import config.ConnectionConfig as cc
from pyspark.sql.functions import expr, lag
from pyspark.sql.window import Window

# SetUp

In [2]:
cc.setupEnvironment()
spark = cc.startLocalCluster("DIM_USER", 4)
spark.getActiveSession()

25/03/18 23:55:01 WARN Utils: Your hostname, 4L3KS-comp resolves to a loopback address: 127.0.1.1; using 192.168.11.127 instead (on interface wlp2s0)
25/03/18 23:55:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/aleks/Downloads/bigtools/spark-3.5.4-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/aleks/.ivy2/cache
The jars for the packages stored in: /home/aleks/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
org.elasticsearch#elasticsearch-spark-30_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-df773ed1-1767-40e6-9b68-0bac272adabc;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.

# Extract

In [5]:
cc.set_connectionProfile("default")
df_users = spark.read.format("jdbc")\
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 10000) \
    .load()

df_subscriptions = spark.read.format("jdbc")\
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscriptions") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptionid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 10000) \
    .load()
df_subscriptions_types = spark.read.format("jdbc")\
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscription_types") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptiontypeid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 10000) \
    .load()

# Transform

In [6]:
#Ok so this joins the two subscriptions tables and then it creates that cute little expression, where for each day, month or year, it uses the date_add function with a start from validfrom. In theory, this should properly calculate the end date, give or take a few days.
df_subscriptions = df_subscriptions.join(df_subscriptions_types, "subscriptiontypeid", "left")
df_subscriptions = df_subscriptions.withColumn(
    "end_date",
    expr("""
        CASE
            WHEN description = 'DAG' THEN date_add(validfrom, 1)
            WHEN description = 'MAAND' THEN date_add(validfrom, 30)
            WHEN description = 'JAAR' THEN date_add(validfrom, 365)
        END
    """)
)

df_joined = df_users.join(df_subscriptions, "userid", "left")

In [10]:
#TRANSFORM
#SCD stuff
#Ok so, im writing this one down so I don't fucking forget
#This bitch organizes the data by user and the date when their address became valid(i.e. validfrom). The for each person it checks what their next address is going to be using lead() and then compare the current one to the "next" one. If any part of it changes, it's marked as a new address. Additionally the end date of the current address is set to one date before the new one starts.
#This only works if there's multiple user instances, which, upon further consideration, is not the case
# window_spec = Window.partitionBy("userid").orderBy("validfrom")
# df_dim_user = df_joined.withColumn("prev_street", lead("street").over(window_spec))\
#     .withColumn("prev_number", lead("number").over(window_spec))\
#     .withColumn("prev_zipcode", lead("zipcode").over(window_spec))\
#     .withColumn("prev_city", lead("city").over(window_spec))\
#     .withColumn("prev_country_code", lead("country_code").over(window_spec))\
#     .withColumn("next_start_date", lead("validfrom").over(window_spec))\
#     .withColumn("is_new_address", expr("""
#         (street <> prev_street) OR
#         (number <> prev_number) OR
#         (zipcode <> prev_zipcode) OR
#         (city <> prev_city) OR
#         (country_code <> prev_country_code)
#     """))\
#     .withColumn("final_end_date", expr("""
#         CASE WHEN is_new_address THEN date_sub(next_start_date, 1)
#         ELSE '9999-12-31' END
#     """))\
#     .drop(
#         "prev_street", "prev_number", "prev_zipcode", "prev_city",
#         "prev_country_code", "next_start_date", "is_new_address"
#     )

RuntimeError: SparkContext or SparkSession should be created first.

In [ ]:
#This bitch works as the previous one, but it actually compares to any previous row/address
#idk anymore, kms
window_spec = Window.partitionBy("userid").orderBy("validfrom")
df_dim_user = df_joined\
    .withColumn("prev_street", lag("street").over(window_spec))\
    .withColumn("prev_number", lag("number").over(window_spec))\
    .withColumn("prev_zipcode", lag("zipcode").over(window_spec))\
    .withColumn("prev_city", lag("city").over(window_spec))\
    .withColumn("prev_country_code", lag("country_code").over(window_spec))\
    .withColumn("prev_validfrom", lag("validfrom").over(window_spec))\
    .withColumn("is_new_address", expr("""
        (street <> prev_street) OR
        (number <> prev_number) OR
        (zipcode <> prev_zipcode) OR
        (city <> prev_city) OR
        (country_code <> prev_country_code)
    """))\
    .withColumn("final_end_date", expr("""
        CASE WHEN is_new_address THEN date_sub(validfrom, 1)
        ELSE '9999-12-31' END
    """))\
    .drop(
        "prev_street", "prev_number", "prev_zipcode", "prev_city",
        "prev_country_code", "prev_validfrom", "is_new_address"
    )

# LOAD

In [8]:
df_dim_user.write.format("delta").mode("overwrite").saveAsTable("dim_user")

In [9]:
spark.stop()